# GRAPES

**Paper:** Younesian et al., *GRAPES: Learning to Sample Graphs for Scalable Graph Neural Networks*, TMLR 2024, [arXiv:2310.03399](https://arxiv.org/abs/2310.03399)

## Kiểm tra GPU

In [2]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    raise RuntimeError('Chưa bật GPU!')

PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
VRAM: 15.6 GB


## Clone repo

In [3]:
!git clone https://github.com/dfdazac/grapes.git
%cd grapes
!ls

Cloning into 'grapes'...
remote: Enumerating objects: 1143, done.
remote: Counting objects: 100% (224/224), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 1143 (delta 141), reused 171 (delta 112), pack-reused 919 (from 1)
Receiving objects: 100% (1143/1143), 230.85 KiB | 13.58 MiB/s, done.
Resolving deltas: 100% (716/716), done.
/content/grapes
analysis  environment.yml  full-batch.py    graphsaint.py  modules
configs   eval.py	   grapes-logo.png  main.py	   README.md


## Cài dependencies

In [4]:
import torch
TORCH = torch.__version__.split('+')[0]
CUDA  = 'cu' + torch.version.cuda.replace('.', '')[:3] if torch.cuda.is_available() else 'cpu'
PYG_URL = f'https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html'
print(f'torch={TORCH}, cuda={CUDA}')

!pip install typed-argument-parser==1.8.0 ogb==1.3.6 wandb psutil tqdm -q
!pip install torch-geometric -q
!pip install torch-scatter torch-sparse -f {PYG_URL} -q

import torch_geometric, ogb, torch_sparse, tap, psutil, wandb
print('torch_geometric:', torch_geometric.__version__)
print('ogb:', ogb.__version__)
print('Dependencies OK')

torch=2.11.0, cuda=cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 9.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 74.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 110.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 142.1 MB/s eta 0:00:00
torch_geometric: 2.7.0
ogb: 1.3.6
Dependencies OK


## Patch compatibility

- **Patch 1–2** (`modules/utils.py`): scipy >= 1.14 không nhận `torch.Tensor` làm index sparse matrix → thêm `.numpy()`
- **Patch 3** (`ogb/nodeproppred/dataset_pyg.py`): PyTorch >= 2.6 đổi default `weights_only=True` → thêm `weights_only=False`

In [5]:
import re, glob

# ── Patch 1 & 2: modules/utils.py ──
utils_path = '/content/grapes/modules/utils.py'
with open(utils_path, 'r') as f:
    lines = f.readlines()
for i, line in enumerate(lines):
    if 'adjacency[nodes].tocoo()' in line:
        lines[i] = line.replace('adjacency[nodes]', 'adjacency[nodes.numpy()]')
        print(f'Patched utils.py line {i+1}: get_neighborhoods')
    if 'row_slice = adjacency[rows]' in line:
        lines[i] = line.replace('adjacency[rows]', 'adjacency[rows.numpy()]')
        print(f'Patched utils.py line {i+1}: slice_adjacency rows')
    if 'row_col_slice = row_slice[:, cols]' in line:
        lines[i] = line.replace('row_slice[:, cols]', 'row_slice[:, cols.numpy()]')
        print(f'Patched utils.py line {i+1}: slice_adjacency cols')
with open(utils_path, 'w') as f:
    f.writelines(lines)

# ── Patch 3 & 5: ogb/nodeproppred/dataset_pyg.py ──
ogb_files = glob.glob('/usr/local/lib/python3.*/dist-packages/ogb/nodeproppred/dataset_pyg.py')
assert ogb_files, 'Không tìm thấy dataset_pyg.py'
ogb_path = ogb_files[0]
with open(ogb_path, 'r') as f:
    ogb_lines = f.readlines()

for i, line in enumerate(ogb_lines):
    # Patch 3: torch.load → weights_only=False
    if 'torch.load(' in line and 'weights_only' not in line:
        ogb_lines[i] = re.sub(
            r'torch\.load\(([^)]+)\)',
            lambda m: f'torch.load({m.group(1)}, weights_only=False)',
            line
        )
        print(f'Patched dataset_pyg.py line {i+1}: weights_only=False')
    # Patch 5: thay cả dòng if input(...) == 'y': → if False:
    if 'input(' in line and 'update' in line and "== 'y'" in line:
        indent = len(line) - len(line.lstrip())
        ogb_lines[i] = ' ' * indent + 'if False:  # patched: skip update prompt\n'
        print(f'Patched dataset_pyg.py line {i+1}: skip update prompt')

with open(ogb_path, 'w') as f:
    f.writelines(ogb_lines)

# ── Patch 4: ogb/utils/url.py — decide_download luôn True ──
url_files = glob.glob('/usr/local/lib/python3.*/dist-packages/ogb/utils/url.py')
assert url_files, 'Không tìm thấy ogb/utils/url.py'
url_path = url_files[0]
with open(url_path, 'r') as f:
    url_lines = f.readlines()
for i, line in enumerate(url_lines):
    if 'def decide_download' in line and 'patched' not in line:
        url_lines.insert(i + 1, '    return True  # patched: always download\n')
        print(f'Patched url.py line {i+1}: decide_download always True')
        break
with open(url_path, 'w') as f:
    f.writelines(url_lines)

# ── Verify syntax ──
import py_compile
for path in [utils_path, ogb_path, url_path]:
    try:
        py_compile.compile(path, doraise=True)
        print(f'  OK syntax: {path.split("/")[-1]}')
    except py_compile.PyCompileError as e:
        print(f'  SYNTAX ERROR: {e}')

print('\nAll patches done.')

Patched utils.py line 78: get_neighborhoods
Patched utils.py line 89: slice_adjacency rows
Patched utils.py line 90: slice_adjacency cols
Patched dataset_pyg.py line 55: skip update prompt
Patched dataset_pyg.py line 69: weights_only=False
Patched dataset_pyg.py line 79: weights_only=False
Patched url.py line 11: decide_download always True
  OK syntax: utils.py
  OK syntax: dataset_pyg.py
  OK syntax: url.py

All patches done.


## Hàm chạy thực nghiệm

In [6]:
import subprocess, re, json, time, datetime
import numpy as np

RESULTS = {}

def run_grapes(config_file, label, runs=3):
    print(f'\n{"="*60}')
    print(f'  {label}  |  {config_file}  |  runs={runs}')
    print(f'{"="*60}')
    cmd = [
        'python', 'main.py',
        f'--config_file={config_file}',
        f'--runs={runs}',
        '--log_wandb=False',
    ]
    print('CMD:', ' '.join(cmd))
    t0 = time.time()
    out = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        cwd='/content/grapes',
        input='n\ny\nn\ny\n',
    )
    elapsed = time.time() - t0

    if out.stdout:
        print(out.stdout[-4000:] if len(out.stdout) > 4000 else out.stdout)
    if out.returncode != 0:
        print('--- STDERR ---')
        print(out.stderr[-2000:])
        return None

    # Parse "Acc: XX.XX ± XX.XX"
    for line in reversed(out.stdout.splitlines()):
        m = re.search(r'Acc:\s*([\d.]+)\s*[±\+\-]+\s*([\d.]+)', line)
        if m:
            mean = float(m.group(1)) / 100
            std  = float(m.group(2)) / 100
            print(f'\n  ✓ {label}: {mean:.4f} ± {std:.4f}  ({elapsed:.0f}s)')
            return {'mean': mean, 'std': std, 'time_s': round(elapsed, 1)}

    # Fallback: dòng test_f1
    for line in reversed(out.stdout.splitlines()):
        if any(k in line.lower() for k in ['test_f1', 'test_accuracy']):
            nums = re.findall(r'\d+\.\d+', line)
            if nums:
                mean = float(nums[-1])
                print(f'\n  ✓ {label} (fallback): {mean:.4f}  ({elapsed:.0f}s)')
                return {'mean': mean, 'std': 0.0, 'time_s': round(elapsed, 1)}

    print('  ✗ Không parse được kết quả.')
    return None

## Cora

In [7]:
r = run_grapes('configs/gflownet/cora.txt', 'GRAPES / Cora', runs=3)
if r: RESULTS['cora_grapes'] = r

r = run_grapes('configs/random/cora.txt', 'Random / Cora', runs=3)
if r: RESULTS['cora_random'] = r

print('\n=== Kết quả hiện tại ===')
print(json.dumps(RESULTS, indent=2))


  GRAPES / Cora  |  configs/gflownet/cora.txt  |  runs=3
CMD: python main.py --config_file=configs/gflownet/cora.txt --runs=3 --log_wandb=False
Memory point 1: 74.92625325520834 MB ± 5.05
Memory point 2: 67.18730078125 MB ± 5.46
Memory point 2: 65.768078125 MB ± 5.45
Acc: 87.10 ± 0.17


  ✓ GRAPES / Cora: 0.8710 ± 0.0017  (56s)

  Random / Cora  |  configs/random/cora.txt  |  runs=3
CMD: python main.py --config_file=configs/random/cora.txt --runs=3 --log_wandb=False
Memory point 1: nan MB ± nan
Memory point 2: nan MB ± nan
Memory point 2: 28.995434244791667 MB ± 0.46
Acc: 86.77 ± 0.38


  ✓ Random / Cora: 0.8677 ± 0.0038  (31s)

=== Kết quả hiện tại ===
{
  "cora_grapes": {
    "mean": 0.871,
    "std": 0.0017000000000000001,
    "time_s": 56.3
  },
  "cora_random": {
    "mean": 0.8676999999999999,
    "std": 0.0038,
    "time_s": 30.5
  }
}


## CiteSeer

In [8]:
r = run_grapes('configs/gflownet/citeseer.txt', 'GRAPES / CiteSeer', runs=3)
if r: RESULTS['citeseer_grapes'] = r

r = run_grapes('configs/random/citeseer.txt', 'Random / CiteSeer', runs=3)
if r: RESULTS['citeseer_random'] = r

print('\n=== Kết quả hiện tại ===')
print(json.dumps(RESULTS, indent=2))


  GRAPES / CiteSeer  |  configs/gflownet/citeseer.txt  |  runs=3
CMD: python main.py --config_file=configs/gflownet/citeseer.txt --runs=3 --log_wandb=False
Memory point 1: 136.41217732747396 MB ± 11.52
Memory point 2: 124.24456746419271 MB ± 19.00
Memory point 2: 120.61967122395833 MB ± 18.99
Acc: 78.57 ± 0.71


  ✓ GRAPES / CiteSeer: 0.7857 ± 0.0071  (95s)

  Random / CiteSeer  |  configs/random/citeseer.txt  |  runs=3
CMD: python main.py --config_file=configs/random/citeseer.txt --runs=3 --log_wandb=False
Memory point 1: nan MB ± nan
Memory point 2: nan MB ± nan
Memory point 2: 45.05152180989583 MB ± 2.72
Acc: 79.00 ± 0.82


  ✓ Random / CiteSeer: 0.7900 ± 0.0082  (56s)

=== Kết quả hiện tại ===
{
  "cora_grapes": {
    "mean": 0.871,
    "std": 0.0017000000000000001,
    "time_s": 56.3
  },
  "cora_random": {
    "mean": 0.8676999999999999,
    "std": 0.0038,
    "time_s": 30.5
  },
  "citeseer_grapes": {
    "mean": 0.7857,
    "std": 0.0070999999999999995,
    "time_s": 94.6
  },

## ogbn-products

In [9]:
r = run_grapes('configs/gflownet/ogbn-products.txt', 'GRAPES / ogbn-products', runs=1)
if r: RESULTS['ogbn_products_grapes'] = r


  GRAPES / ogbn-products  |  configs/gflownet/ogbn-products.txt  |  runs=1
CMD: python main.py --config_file=configs/gflownet/ogbn-products.txt --runs=1 --log_wandb=False
Extracting /content/grapes/data/products/OGB/products.zip
Loading necessary files...
This might take a while.
Processing graphs...
Converting graphs into PyG objects...
Saving...

--- STDERR ---
ss_gfn=2.23e+5, batch_loss_c=0.725, log_z=7.06, log_probs=-1.15e+4]
Epoch 9: 100%|██████████| 769/769 [04:03<00:00,  3.15it/s, batch_loss_gfn=2.23e+7, batch_loss_c=0.00518, log_z=9.42, log_probs=-4.81e+3]
09:44:16 - INFO - root - Evaluating



## ogbn-arxiv

In [10]:
r = run_grapes('configs/gflownet/ogbn-arxiv.txt', 'GRAPES / ogbn-arxiv', runs=3)
if r: RESULTS['ogbn_arxiv_grapes'] = r

r = run_grapes('configs/random/ogbn-arxiv.txt', 'Random / ogbn-arxiv', runs=3)
if r: RESULTS['ogbn_arxiv_random'] = r

print('\n=== Kết quả hiện tại ===')
print(json.dumps(RESULTS, indent=2))


  GRAPES / ogbn-arxiv  |  configs/gflownet/ogbn-arxiv.txt  |  runs=3
CMD: python main.py --config_file=configs/gflownet/ogbn-arxiv.txt --runs=3 --log_wandb=False
Extracting /content/grapes/data/ogbn-arxiv/OGB/arxiv.zip
Loading necessary files...
This might take a while.
Processing graphs...
Converting graphs into PyG objects...
Saving...
Memory point 1: 327.5443982436505 MB ± 10.70
Memory point 2: 59.496133547055436 MB ± 9.53
Memory point 2: 59.25541152173065 MB ± 9.53
Acc: 62.04 ± 0.31


  ✓ GRAPES / ogbn-arxiv: 0.6204 ± 0.0031  (11114s)

  Random / ogbn-arxiv  |  configs/random/ogbn-arxiv.txt  |  runs=3
CMD: python main.py --config_file=configs/random/ogbn-arxiv.txt --runs=3 --log_wandb=False
Memory point 1: nan MB ± nan
Memory point 2: nan MB ± nan
Memory point 2: 18.688361478084037 MB ± 0.12
Acc: 61.28 ± 0.29


  ✓ Random / ogbn-arxiv: 0.6128 ± 0.0029  (8133s)

=== Kết quả hiện tại ===
{
  "cora_grapes": {
    "mean": 0.871,
    "std": 0.0017000000000000001,
    "time_s": 56.3
  }

## Tổng hợp

In [ ]:
import json, datetime, torch

print(f'\n{"="*60}')
print(f'{"Dataset":<22} {"Method":<10} {"Acc":>10} {"± Std":>8}')
print(f'{"-"*60}')
for key, val in RESULTS.items():
    ds, method = key.rsplit('_', 1)
    print(f'{ds:<22} {method:<10} {val["mean"]:>10.4f} {val["std"]:>8.4f}')
print(f'{"="*60}')

output = {
    'timestamp': datetime.datetime.now().isoformat(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'torch_version': torch.__version__,
    'results': RESULTS
}
with open('/content/grapes/grapes_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print('\nSaved: grapes_results.json')

## Tải file về máy

In [12]:
from google.colab import files
files.download('/content/grapes/grapes_results.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>